# D-Fire 팀 학습 준비와 실행 — YOLO11n
두 사람 모두 같은 모델·fold 1·seed 42를 사용합니다. 각자 이 노트북을 **자기 Drive에 사본 저장**하고 GPU 런타임을 사용하세요. MEMBER_ID와 공유 폴더 경로만 각자 바꿉니다. 모든 학습/시험 실행 스위치는 기본 False입니다. 80epoch는 본인이 명시적으로 켜기 전에는 실행되지 않습니다.

새 본 학습의 기준 모델은 YOLO11n입니다. 과거 YOLO26n 3epoch는 보관용 시험 결과입니다. 일반 COCO 성능을 D-Fire 성능으로 해석하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 각자 설정
본인은 `jongwon`, 팀원은 `member2`처럼 서로 다른 영문 ID를 씁니다. 팀원은 공유 프로젝트 폴더의 바로가기를 **내 드라이브**에 추가하고 실제 경로를 맞추세요. 공유 결과 폴더의 편집 권한도 필요합니다.

각자 독립적인 GPU 런타임을 씁니다. 두 번의 80epoch 결과가 합쳐져 160epoch 모델이 되는 것은 아닙니다. 동일 조건 재현성 확인용이며, 최종 서비스에는 선정된 한 파일만 사용합니다.

In [ ]:
from pathlib import Path

MEMBER_ID = 'jongwon'  # 팀원: 'member2'
DRIVE_ROOT = Path('/content/drive/MyDrive/2026 3학년 2학기 진로탐색')
LABELS_REVIEWED = False
RUN_FULL_TRAIN = False
RUN_VALIDATION = False
RUN_TEST_EVAL = False
FINAL_RUN_SELECTED = False
BATCH = 16  # 메모리 부족이면 새 실행에서 8; 동일 재개는 원래 batch 유지
RESUME_FROM = ''  # 중단된 기존 실행의 Drive 폴더. 새 학습은 빈 문자열
RUN_TO_EVALUATE = ''  # 이미 끝난 실행을 평가할 때 해당 Drive 폴더

SHARED_PROJECT = DRIVE_ROOT / 'Code' / 'fire-ai'
DFIRE = DRIVE_ROOT / 'Data' / 'DFire'
DRIVE_RESULTS = DFIRE / 'training_results'
LOCAL_DATASET = Path('/content/fire-ai-data-v2')
LOCAL_RUNS = Path('/content/fire-ai-runs')
for path in (SHARED_PROJECT / 'scripts' / 'training_workflow.py', DFIRE / 'D-Fire.zip', DFIRE / 'd-fire 텍스트 분할.zip'):
    assert path.is_file(), f'공유 바로가기/경로를 확인하세요: {path}'
print('공유 프로젝트:', SHARED_PROJECT)

## 2. 실행 코드 사본과 패키지
코드는 런타임 로컬로 복사해 공유 파일의 동시 수정 영향을 줄입니다. Colab의 CUDA용 PyTorch를 유지하고 Ultralytics 버전만 고정합니다. 설치 후 런타임 재시작 안내가 뜨면 재시작하고 위 셀부터 다시 실행하세요.

In [ ]:
import shutil
import subprocess
import sys
import uuid

PROJECT_DIR = Path('/content') / ('fire-ai-code-' + uuid.uuid4().hex[:8])
for folder in ('scripts', 'configs', 'notebooks'):
    shutil.copytree(SHARED_PROJECT / folder, PROJECT_DIR / folder, ignore=shutil.ignore_patterns('__pycache__'))
shutil.copy2(SHARED_PROJECT / 'requirements-colab.txt', PROJECT_DIR / 'requirements-colab.txt')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements-colab.txt')], check=True)
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
# Re-running this cell must load the new local copy, not a cached older module.
import importlib
for module_name in ('training_workflow', 'prepare_dfire'):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
from training_workflow import copy_verified, sha256, read_json, preflight, train, evaluate, collect_results, runtime_info, WORKFLOW_VERSION
assert WORKFLOW_VERSION == '2026-09-30-validator-fix-1', '공유 폴더의 코드 동기화를 확인하세요.'
print('Workflow:', WORKFLOW_VERSION)
print(runtime_info())
assert runtime_info()['gpu'], '런타임 유형을 GPU로 변경하세요.'

## 3. 원본 복사와 전체 데이터 검사
압축과 학습 데이터는 Colab 로컬에서 읽고, 결과만 Drive로 보냅니다. 이미지 디코딩·라벨·공식 분할·원본 해시를 검사합니다. 런타임이 바뀌면 다시 실행하세요. 출력 폴더가 불완전하다는 오류는 LOCAL_DATASET 이름을 새로 바꿔서 해결합니다. 원본 ZIP은 수정하지 않습니다.

In [ ]:
LOCAL_ARCHIVES = Path('/content/dfire-archives-v2')
LOCAL_ARCHIVES.mkdir(exist_ok=True)
for filename in ('D-Fire.zip', 'd-fire 텍스트 분할.zip'):
    source, target = DFIRE / filename, LOCAL_ARCHIVES / filename
    if not target.is_file() or sha256(target) != sha256(source):
        print('Copying:', filename, flush=True)
        copy_verified(source, target)
subprocess.run([
    sys.executable, str(PROJECT_DIR / 'scripts' / 'prepare_dfire.py'),
    '--dataset-zip', str(LOCAL_ARCHIVES / 'D-Fire.zip'),
    '--split-zip', str(LOCAL_ARCHIVES / 'd-fire 텍스트 분할.zip'),
    '--output', str(LOCAL_DATASET), '--fold', '1', '--verify-images',
], check=True)
summary = read_json(LOCAL_DATASET / 'dataset_summary.json')
print(summary)
print(preflight(LOCAL_DATASET, DRIVE_RESULTS))

## 4. 사전학습 가중치 로딩 확인 — 학습 아님
공유된 사전학습 파일을 로컬로 복사하고 해시를 확인합니다. 빈 이미지 추론은 설치와 모델 로딩만 검사합니다. 아직 이 모델은 화재 전용 모델이 아닙니다.

In [ ]:
import numpy as np
from ultralytics import YOLO

PRETRAINED = Path('/content/fire-ai-pretrained/yolo11n.pt')
shared_weights = DFIRE / 'pretrained' / 'yolo11n.pt'
manifest = read_json(DFIRE / 'pretrained' / 'yolo11n_manifest.json')
assert sha256(shared_weights) == manifest['sha256'], '공유 가중치가 검사본과 다릅니다.'
copy_verified(shared_weights, PRETRAINED)
probe_model = YOLO(str(PRETRAINED))
probe_model.predict(np.zeros((640,640,3), dtype=np.uint8), device=0, save=False, verbose=False)
del probe_model
print('YOLO11n 로딩 및 GPU 추론 확인 완료. 본 학습은 아직 실행하지 않았습니다.')

## 5. 라벨 시각화
검증 이미지에서 smoke·fire·배경 예시를 각각 표시합니다. 불꽃/연기 위치와 박스가 맞는지 확인하세요. 확인한 뒤 아래 확인 셀만 True로 바꿉니다.

In [ ]:
import csv
import random
import cv2
import matplotlib.pyplot as plt

with (LOCAL_DATASET / 'dataset_manifest_v001.csv').open(encoding='utf-8-sig', newline='') as f:
    val_rows = [r for r in csv.DictReader(f) if r['split'] == 'val']
rng = random.Random(42)
chosen = []
for cls in ('smoke', 'fire', 'background'):
    candidates = [r for r in val_rows if (r['is_negative'] == 'True' if cls == 'background' else cls in r['classes'].split('|'))]
    chosen.extend(rng.sample(candidates, min(3, len(candidates))))
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, row in zip(axes.ravel(), chosen):
    im = cv2.imread(str(LOCAL_DATASET / row['image_path']))
    h, w = im.shape[:2]
    for line in (LOCAL_DATASET / row['label_path']).read_text(encoding='utf-8-sig').splitlines():
        c, x, y, bw, bh = map(float, line.split())
        p1, p2 = (int((x-bw/2)*w), int((y-bh/2)*h)), (int((x+bw/2)*w), int((y+bh/2)*h))
        color = (0,200,255) if int(c)==0 else (0,0,255)
        cv2.rectangle(im, p1, p2, color, 2)
        cv2.putText(im, 'smoke' if int(c)==0 else 'fire', (p1[0], max(18,p1[1])), cv2.FONT_HERSHEY_SIMPLEX, .6, color, 2)
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(row['file_name']); ax.axis('off')
plt.tight_layout()

In [ ]:
LABELS_REVIEWED = False  # 위 라벨을 직접 확인한 다음에만 True
print('라벨 검토:', LABELS_REVIEWED)

## 6. 여기까지가 사전 준비 완료
**오늘 사전 준비만 할 때는 여기서 멈춥니다.** 아래 셀은 기본 False라서 전체 실행을 눌러도 학습하지 않습니다. 나중에 이 셀의 RUN_FULL_TRAIN을 True로 바꿔 실행하세요.

매 epoch 끝에 last.pt·best.pt·로그를 Drive의 개인 실행 폴더에 해시 확인 후 보관합니다. 마지막 완료 epoch까지 복구할 수 있고, 중단된 epoch의 작업은 다시 해야 합니다. 백업 실패는 오류로 표시하고 학습을 중단합니다. 로컬 검증은 Drive 마운트의 읽기/쓰기 확인이며 Google 서버 동기화나 세션 존속을 보장하지는 않습니다.

In [ ]:
RUN_FULL_TRAIN = False  # 두 사람이 각자의 런타임에서 직접 True로 변경
TRAINED_RUN_DIR = None
if RUN_FULL_TRAIN:
    TRAINED_RUN_DIR = train(
        LOCAL_DATASET, DRIVE_RESULTS, LOCAL_RUNS, PRETRAINED, MEMBER_ID,
        approved=RUN_FULL_TRAIN, labels_reviewed=LABELS_REVIEWED,
        batch=BATCH, resume_from=RESUME_FROM or None,
    )
    print('완료된 실행:', TRAINED_RUN_DIR)
else:
    print('학습 OFF. 사전 준비만 완료했습니다.')

## 7. 완료 모델 validation 평가
방금 학습이 완료된 경우 자동으로 검증합니다. 기존 실행을 평가하려면 RUN_TO_EVALUATE에 폴더를 넣고 RUN_VALIDATION=True로 설정합니다. 클래스별 성능, 그래프, 실행환경, 모델 파일이 각 실행 폴더에 남습니다.

In [ ]:
ACTIVE_RUN = TRAINED_RUN_DIR or (Path(RUN_TO_EVALUATE) if RUN_TO_EVALUATE else None)
if ACTIVE_RUN and (TRAINED_RUN_DIR or RUN_VALIDATION):
    validation = evaluate(ACTIVE_RUN, LOCAL_DATASET, LOCAL_RUNS, split='val')
    print(validation)
else:
    print('아직 완료된 학습이 없거나 평가 OFF입니다.')

## 8. 두 사람 결과 확인
공유 파일을 덮어쓰지 않고 실행별 요약을 읽습니다. split_sha256과 pretrained_sha256이 같아야 같은 데이터·초기 가중치입니다. 같은 seed라도 GPU/라이브러리 차이에 따라 완전히 같은 숫자가 나오지는 않을 수 있습니다. 학습 결과의 mAP만 보지 말고 fire/smoke recall과 정상 영상 오탐도 함께 보고 한 실행을 선정하세요.

In [ ]:
import pandas as pd
display(pd.DataFrame(collect_results(DRIVE_RESULTS)))

## 9. 최종 test 평가 — 기본 OFF
두 사람 모두 test를 보며 설정을 수정하지 마세요. validation과 별도 영상 확인으로 최종 실행을 하나 고른 다음 담당자 한 명만 실행합니다. RUN_TO_EVALUATE에는 선정한 실행 폴더를 지정합니다.

In [ ]:
RUN_TEST_EVAL = False
FINAL_RUN_SELECTED = False
if RUN_TEST_EVAL:
    assert FINAL_RUN_SELECTED and RUN_TO_EVALUATE, '팀 최종 모델 선정과 실행 경로가 필요합니다.'
    print(evaluate(Path(RUN_TO_EVALUATE), LOCAL_DATASET, LOCAL_RUNS, split='test', test_approved=True))
else:
    print('test 평가 OFF')

## 중단 후 재개
새 GPU 런타임에서 위 준비 셀을 다시 실행하고, RESUME_FROM에 중단된 Drive 실행 폴더를 입력합니다. 원래 BATCH를 유지하고 라벨 확인 후 본 학습 셀을 True로 실행합니다. 새 실행 폴더에 parent_run을 남기며 원래 결과를 덮어쓰지 않습니다. optimizer·epoch를 복구하지만 난수 상태/early-stopping 대기 횟수 등으로 무중단 실행과 완전히 같지는 않을 수 있습니다.

## 개발 연결
최종 선택 실행의 `final/best.pt`를 사용합니다. `scripts/infer_media.py`는 이미지/MP4의 프레임별 결과 JSONL을 만듭니다. 화재 사건 확정, 시간 필터, 지도 좌표 변환은 별도 개발입니다. 자세한 사용법은 `docs/TRAINING_READY.md`를 확인하세요.